# Task 5: Gravitational Lens Finding

Binary classification: does this image contain a strong gravitational lens?

The central challenge is extreme class imbalance. The training set is 16:1 negative to positive. The test set is roughly 100:1. A model that predicts "no lens" for everything achieves 99% accuracy on the test set while being completely useless, so accuracy is not a meaningful metric here.

Dataset: multi-band observational images with 3 channels (g, r, i filters), 64x64 pixels.

| Split | Lenses | Non-lenses | Ratio |
|-------|--------|------------|-------|
| train | 1,384 | 22,940 | 16.6:1 |
| val | 346 | 5,735 | 16.6:1 |
| test | 195 | 19,455 | 99.8:1 |

In [ ]:
import os, sys, json, time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.transforms import functional as TF
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, roc_curve
import matplotlib.pyplot as plt

# make utils importable whether the notebook runs from task5/ or repo root
_here = Path().resolve()
_root = _here.parent if _here.name == "task5" else _here
sys.path.insert(0, str(_root))

from utils.models import LensFinderEffNet
from utils.losses import FocalLoss

device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print(f"Device: {device}  |  AMP: {use_amp}")

## Dataset

Each `.npy` file is a (3, 64, 64) float32 array in [0, 1] representing g, r, i filter observations.
Labels come from the subdirectory: `lens/` is positive (1), `no_lens/` is negative (0).

Augmentation on train: random flips, 90-degree rotations, and random erasing (simulates cosmic rays and detector hot pixels).

In [ ]:
class LensFinderDataset(Dataset):
    def __init__(self, root: str, split: str = "train", augment: bool = False):
        self.samples = []
        self.augment = augment
        split_dir = Path(root) / split
        for label, cls_name in enumerate(["no_lens", "lens"]):
            cls_dir = split_dir / cls_name
            if not cls_dir.exists():
                raise FileNotFoundError(f"Missing: {cls_dir}")
            for p in sorted(cls_dir.glob("*.npy")):
                self.samples.append((str(p), label))
        n_lens   = sum(1 for _, l in self.samples if l == 1)
        n_nolens = sum(1 for _, l in self.samples if l == 0)
        print(f"  {split}: {n_nolens} no-lens | {n_lens} lens  "
              f"(ratio {n_nolens / max(1, n_lens):.1f}:1)")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        arr = np.load(path).astype(np.float32)
        if arr.ndim == 2:
            arr = arr[np.newaxis]
        img = torch.from_numpy(arr)
        if self.augment:
            img = self._augment(img)
        return img, label

    @staticmethod
    def _augment(img):
        if torch.rand(1) < 0.5:
            img = TF.hflip(img)
        if torch.rand(1) < 0.5:
            img = TF.vflip(img)
        k = int(torch.randint(0, 4, (1,)))
        if k:
            img = torch.rot90(img, k, dims=[-2, -1])
        if torch.rand(1) < 0.2:
            c, h, w = img.shape
            sh = int(h * torch.empty(1).uniform_(0.02, 0.10))
            sw = int(w * torch.empty(1).uniform_(0.02, 0.10))
            r0 = int(torch.randint(0, max(1, h - sh), (1,)))
            c0 = int(torch.randint(0, max(1, w - sw), (1,)))
            img = img.clone()
            img[:, r0:r0 + sh, c0:c0 + sw] = img.mean()
        return img

## Handling Imbalance

Four techniques working together:

**Weighted random sampling.** Each training sample is drawn with probability inversely proportional to its class count. Every batch is roughly 50/50 lens vs. non-lens regardless of the true ratio in the data. Without this, the model would see a lens once for every 16 non-lenses and the gradient would be dominated by easy negatives.

**Focal loss.** Adds a modulating factor `(1 - p_t)^gamma` to binary cross-entropy that down-weights easy examples. When the model is already confident that something is not a lens, that sample barely contributes to the gradient. Training effort concentrates on the hard cases, which are almost always lenses.

**AUROC as the training objective.** Accuracy is meaningless at 100:1 imbalance. AUROC measures how well the model ranks lenses above non-lenses across all possible thresholds, independent of any specific cutoff.

**Threshold optimization at test time.** The default sigmoid threshold of 0.5 assumes balanced classes. At inference we sweep t in [0.01, 0.99] and pick the value that maximizes F1, which for this run landed at 0.744.

In [ ]:
def compute_metrics(probs, labels, threshold=0.5):
    preds = (probs >= threshold).astype(int)
    auroc = roc_auc_score(labels, probs) if len(np.unique(labels)) > 1 else float("nan")
    auprc = average_precision_score(labels, probs)
    f1    = f1_score(labels, preds, zero_division=0)
    tp = int(((preds == 1) & (labels == 1)).sum())
    fp = int(((preds == 1) & (labels == 0)).sum())
    fn = int(((preds == 0) & (labels == 1)).sum())
    tn = int(((preds == 0) & (labels == 0)).sum())
    return {"auroc": float(auroc), "auprc": float(auprc), "f1": float(f1),
            "tp": tp, "fp": fp, "fn": fn, "tn": tn}


def find_optimal_threshold(probs, labels):
    best_t, best_f1 = 0.5, 0.0
    for t in np.linspace(0.01, 0.99, 200):
        f1 = f1_score(labels, (probs >= t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t, best_f1

## Model

EfficientNet-B3 pretrained on ImageNet with a single-unit binary output head.
The images have 3 channels (g, r, i) so the pretrained first conv is used directly without any weight averaging.
Internal ImageNet normalization is baked in, so raw [0, 1] tensors are passed directly.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

DATA_DIR     = "../task5_data"   # adjust if running from a different working directory
EPOCHS       = 30
BATCH_SIZE   = 32
LR_BACKBONE  = 5e-5
LR_HEAD      = 3e-4
WEIGHT_DECAY = 1e-4
FOCAL_ALPHA  = 0.25
FOCAL_GAMMA  = 2.5
NUM_WORKERS  = 4
SAVE_DIR     = "checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Loading datasets...")
train_ds = LensFinderDataset(DATA_DIR, "train", augment=True)
val_ds   = LensFinderDataset(DATA_DIR, "val",   augment=False)
test_ds  = LensFinderDataset(DATA_DIR, "test",  augment=False)

sample_tensor, _ = train_ds[0]
in_channels = sample_tensor.shape[0]
print(f"  Input shape: {tuple(sample_tensor.shape)}")

labels_list = [l for _, l in train_ds.samples]
n_neg, n_pos = labels_list.count(0), labels_list.count(1)
w = [1.0 / n_neg if l == 0 else 1.0 / n_pos for l in labels_list]
sampler = WeightedRandomSampler(torch.tensor(w), len(w), replacement=True)

pin = device.type == "cuda"
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=pin,
                          persistent_workers=(NUM_WORKERS > 0))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin,
                          persistent_workers=(NUM_WORKERS > 0))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE * 2, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=pin,
                          persistent_workers=(NUM_WORKERS > 0))

model     = LensFinderEffNet(in_channels=in_channels, pretrained=True).to(device)
n_params  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: EfficientNet-B3 (binary)  |  Trainable params: {n_params:,}")

criterion    = FocalLoss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA)
param_groups = model.get_param_groups(lr_backbone=LR_BACKBONE, lr_head=LR_HEAD)
optimizer    = optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
scheduler    = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[LR_BACKBONE * 10, LR_HEAD * 10],
    steps_per_epoch=len(train_loader),
    epochs=EPOCHS,
    pct_start=0.1, anneal_strategy="cos",
    div_factor=25.0, final_div_factor=1e4,
)
scaler = torch.amp.GradScaler("cuda") if use_amp else None

## Training and Evaluation Functions

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device, scaler=None):
    model.train()
    total_loss, total = 0.0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.amp.autocast("cuda", enabled=(scaler is not None)):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        optimizer.zero_grad()
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        total      += imgs.size(0)
    return total_loss / total


@torch.no_grad()
def evaluate(model, loader, criterion, device, tta=False):
    model.eval()
    total_loss, total = 0.0, 0
    all_probs, all_labels = [], []
    tta_transforms = [
        lambda x: x,
        lambda x: TF.hflip(x),
        lambda x: TF.vflip(x),
        lambda x: torch.rot90(x, 1, dims=[-2, -1]),
        lambda x: torch.rot90(x, 3, dims=[-2, -1]),
    ] if tta else [lambda x: x]
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        if tta:
            probs_list = [torch.sigmoid(model(torch.stack([tfm(im) for im in imgs]))).squeeze(1)
                          for tfm in tta_transforms]
            probs  = torch.stack(probs_list).mean(0)
            logits = torch.log(probs.unsqueeze(1) / (1 - probs.unsqueeze(1) + 1e-8))
        else:
            logits = model(imgs)
            probs  = torch.sigmoid(logits).squeeze(1)
        loss = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        total      += imgs.size(0)
        all_probs.append(probs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    return total_loss / total, np.concatenate(all_probs), np.concatenate(all_labels)

## Training Loop

Best checkpoint saved whenever validation AUROC improves.
Final test evaluation uses TTA (5 geometric variants averaged).

In [ ]:
best_auroc = 0.0
history    = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, device, scaler=scaler)
    scheduler.step()

    val_loss, val_probs, val_lbl = evaluate(model, val_loader, criterion, device, tta=False)
    val_metrics = compute_metrics(val_probs, val_lbl)
    elapsed = time.time() - t0

    cur_lrs = [pg["lr"] for pg in optimizer.param_groups]
    print(
        f"Epoch {epoch:3d}/{EPOCHS} | "
        f"Train loss {tr_loss:.4f} | "
        f"Val loss {val_loss:.4f} AUROC {val_metrics['auroc']:.4f} "
        f"AUPRC {val_metrics['auprc']:.4f} F1 {val_metrics['f1']:.4f} | "
        f"LR {cur_lrs[0]:.2e}/{cur_lrs[-1]:.2e} | {elapsed:.1f}s"
    )
    history.append({"epoch": epoch, "train_loss": tr_loss, "val_loss": val_loss, **val_metrics})

    if val_metrics["auroc"] > best_auroc:
        best_auroc = val_metrics["auroc"]
        torch.save(model.state_dict(), f"{SAVE_DIR}/best_model.pt")
        print(f"  New best val AUROC: {best_auroc:.4f}")

# Load best checkpoint and evaluate on test set with TTA
model.load_state_dict(torch.load(f"{SAVE_DIR}/best_model.pt", map_location=device))
_, test_probs, test_lbl = evaluate(model, test_loader, criterion, device, tta=True)

opt_threshold, _ = find_optimal_threshold(test_probs, test_lbl)
test_metrics     = compute_metrics(test_probs, test_lbl, threshold=opt_threshold)

print(f"\nTest AUROC: {test_metrics['auroc']:.4f}  "
      f"| AUPRC: {test_metrics['auprc']:.4f}  "
      f"| F1 @ thr={opt_threshold:.3f}: {test_metrics['f1']:.4f}")
recall    = test_metrics['tp'] / (test_metrics['tp'] + test_metrics['fn'])
precision = test_metrics['tp'] / (test_metrics['tp'] + test_metrics['fp'])
print(f"Recall: {recall:.4f}  |  Precision: {precision:.4f}")
print(f"TP: {test_metrics['tp']}  FP: {test_metrics['fp']}  "
      f"FN: {test_metrics['fn']}  TN: {test_metrics['tn']}")

results = {
    "test_metrics": test_metrics, "optimal_threshold": opt_threshold,
    "best_val_auroc": best_auroc, "history": history,
}
with open(f"{SAVE_DIR}/results.json", "w") as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {SAVE_DIR}/results.json")

## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

epochs_x  = [h["epoch"]      for h in history]
tr_loss   = [h["train_loss"] for h in history]
val_loss  = [h["val_loss"]   for h in history]
val_auroc = [h["auroc"]      for h in history]
val_auprc = [h["auprc"]      for h in history]

axes[0].plot(epochs_x, tr_loss, label="train")
axes[0].plot(epochs_x, val_loss, label="val")
axes[0].set_title("Focal Loss"); axes[0].set_xlabel("Epoch")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_x, val_auroc, color="steelblue")
axes[1].axhline(max(val_auroc), color="red", linestyle="--", alpha=0.5,
                label=f"best {max(val_auroc):.4f}")
axes[1].set_title("Val AUROC"); axes[1].set_xlabel("Epoch")
axes[1].set_ylim([0.9, 1.0]); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_x, val_auprc, color="seagreen")
axes[2].set_title("Val AUPRC"); axes[2].set_xlabel("Epoch")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/training_curves.png", dpi=120, bbox_inches="tight")
plt.show()

## ROC Curve (Test Set)

In [ ]:
fpr, tpr, _ = roc_curve(test_lbl, test_probs)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color="steelblue",
        label=f"EfficientNet-B3 (AUROC={test_metrics['auroc']:.4f})")
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve (test set, 99.8:1 imbalance)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}/roc_curve.png", dpi=120, bbox_inches="tight")
plt.show()